# Random forest by hand
Three decision trees, each trained on a different random subset of a small dataset, then a majority vote.
We build the subsets three ways: row sampling, column sampling, and both together.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text

## 1. The data: 100 rows, 5 input columns, 1 output column

In [ ]:
# 5 informative columns, two classes
X, y = make_classification(n_samples=100, n_features=5, n_redundant=0, n_informative=5,
                           n_clusters_per_class=1, random_state=4)
df = pd.DataFrame(X, columns=["col1", "col2", "col3", "col4", "col5"]).round(3)
df["target"] = y
print(df.shape)
df.head()

## 2. Three sampling functions
`rng` is a seeded random generator, so every run draws the same samples.

In [ ]:
rng = np.random.default_rng(4)


def sample_rows(df, share):
    """A random share of the rows, drawn with replacement (rows can repeat)."""
    n = int(share * len(df))
    return df.sample(n, replace=True, random_state=rng.integers(10**9))


def sample_features(df, share):
    """All rows, a random share of the input columns (no repeats); the target column is always kept."""
    n = int(share * (df.shape[1] - 1))           # do not count the target column
    cols = list(rng.choice(df.columns[:-1], n, replace=False))
    return df[cols + ["target"]]


def combined_sampling(df, row_share, col_share):
    """Row sampling first, then column sampling on the result."""
    return sample_features(sample_rows(df, row_share), col_share)

## 3. The query point
We ask every tree about row 7 of the data. Its true class is 0.

In [ ]:
query = df.iloc[[7]]
query

In [ ]:
def forest_by_hand(make_subset):
    """Train three fully grown trees, each on its own random subset, and collect their votes for the query."""
    votes = []
    for k in range(1, 4):
        d = make_subset()
        inputs = d.columns[:-1]                        # the input columns this tree got
        tree = DecisionTreeClassifier(random_state=0).fit(d[inputs], d["target"])
        vote = int(tree.predict(query[inputs])[0])     # the query, with the same columns
        votes.append(vote)
        print(f"tree {k}: {d.shape[0]} rows ({d.index.duplicated().sum()} repeats), "
              f"columns {list(inputs)}, depth {tree.get_depth()}, vote {vote}")
    print("votes:", votes, "-> majority:", max(set(votes), key=votes.count))
    return tree, inputs

## 4. Row sampling: each tree gets 20% of the rows (20 rows), drawn with replacement

In [ ]:
tree, inputs = forest_by_hand(lambda: sample_rows(df, 0.2))
# the last tree, printed as text (feature_names maps X[0], X[1]... to real column names)
print(export_text(tree, feature_names=list(inputs)))

## 5. Column sampling: each tree gets every row but 80% of the columns (4 of 5)

In [ ]:
tree, inputs = forest_by_hand(lambda: sample_features(df, 0.8))

## 6. Combined sampling: 50% of the rows and 50% of the columns (2 of 5)

In [ ]:
tree, inputs = forest_by_hand(lambda: combined_sampling(df, 0.5, 0.5))
print(export_text(tree, feature_names=list(inputs)))

## 7. The same in one line with scikit-learn
`RandomForestClassifier` does all of this for us, with 100 trees by default.

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(df.iloc[:, :5], df["target"])
print("number of trees:", len(rf.estimators_))
print("prediction for the query:", rf.predict(query.iloc[:, :5]))